# Coffee Station · Gemini + Mock Agent 실습

실제 DB 대신 **가상 카페 4개·판매 옵션 10개**만 사용합니다. `RUN_LIVE=False`가 기본이므로 전체 실행은 API를 호출하지 않습니다. 실제 호출은 `True`로 바꾼 후 진행하세요. 호출 비용/할당량이 사용될 수 있습니다.

이번 구현: 직접 Agent Loop, 수동 Function Calling, 조건 보호, 실행 trace. HTTP·Express·MySQL은 미구현입니다. 자연어의 필수 조건 자동 확정도 아직 구현하지 않았습니다. 아래 `constraints`는 사용자가 확인한 조건으로 취급하므로, 문장을 수정할 때 함께 수정하세요.

VS Code에서 `ai-server/.venv/Scripts/python.exe`를 커널로 선택합니다. 패키지가 없다면 터미널에서 `uv pip install --python ai-server/.venv/Scripts/python.exe -r ai-server/requirements-notebook.txt`를 실행하세요. pip 사용 시에는 해당 환경에 pip를 먼저 설치해야 합니다.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
AI_ROOT = next((p for base in (cwd, *cwd.parents)
                for p in (base, base / 'ai-server')
                if (p / 'app/repositories/mock.py').is_file()), None)
if AI_ROOT is None:
    raise RuntimeError('coffee-station 또는 ai-server 안에서 노트북을 실행하세요.')
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

import json
from dataclasses import asdict, replace
from app.repositories.base import Location, SearchFilters
from app.repositories.mock import MockRepository
from app.agent.loop import run_turn
from app.agent.state import ConversationState
from app.llm.base import ScriptedModel, Reply, ToolCall

repo = MockRepository()
assert all(m.offer.evidence.is_mock for m in repo.search_menus(SearchFilters()))
print('Mock catalog ready. DB 연결 없음.')

## 1. 임시 데이터와 결과 표시

맛·공부 목적 태그도 학습용 설정값입니다. 실제 매장 정보나 무설탕 여부를 의미하지 않습니다. 거리는 직선거리이며 도보 시간이 아닙니다.

In [ ]:
def show_result(result):
    print(result.status, result.message)
    for item in result.recommendations:
        print(json.dumps({
            'offer_id': item.offer.id, 'cafe': item.cafe.name,
            'menu': item.offer.name, 'price_krw': item.offer.price_krw,
            'size': item.offer.size, 'temperature': item.offer.temperature,
            'distance_m': None if item.distance_m is None else round(item.distance_m),
            'is_mock': item.offer.evidence.is_mock,
        }, ensure_ascii=False))
    print('LLM 호출:', result.llm_calls, '| 총 시간(ms):', result.elapsed_ms)
    for event in result.trace:
        print(event)

for item in repo.search_menus(SearchFilters(sort_by='price')):
    print(item.offer.id, item.cafe.name, item.offer.name, item.offer.price_krw)

## 2. 네트워크 없는 Agent Loop 점검

정해진 모델 응답으로 도구 호출 → 실행 → 결과 관찰 → 최종 검증 흐름을 확인합니다.

In [ ]:
origin = Location(37.63, 126.83)  # 가상 위치
constraints = SearchFilters(origin=origin, radius_m=500, max_price_krw=5000,
                            sweetness='NOT_SWEET', sort_by='price')
fake = ScriptedModel([
    Reply(calls=(ToolCall('search_menus', {}),)),
    Reply(text=json.dumps({'type': 'RESULT', 'offer_ids': ['offer-2', 'offer-1', 'offer-6']})),
])
offline_result = run_turn(fake, repo, '근처에서 5천 원 이하의 달지 않은 커피 추천해 줘.',
                          constraints=constraints)
show_result(offline_result)
assert offline_result.status == 'RESULT'
assert [m.offer.price_krw for m in offline_result.recommendations] == [3000, 3500, 5000]

## 3. 실제 Gemini 호출 설정

이 셀은 입력 대기나 API 호출 없이 `.env` 설정만 확인합니다. 저장소 루트 또는 `ai-server/.env`에 `GEMINI_API_KEY`와 `GEMINI_MODEL`을 설정하세요. 키를 노트북 코드에 적지 마세요. 우선순위는 환경변수 > ai-server/.env > 저장소 .env입니다.

`RUN_LIVE=True`일 때 누락된 항목이 있으면 즉시 오류로 알려줍니다. 기존에 테스트한 모델 ID를 사용하세요. `.env` 변경 후에는 이 셀만 다시 실행하면 됩니다. 키 값은 출력하지 않습니다.

실제 모델 초기화와 호출은 **4번 셀**에서 시작합니다. 모델 호출 최대 6회·도구 호출 최대 10회·턴 시간 예산 60초입니다. 기존 숨김 입력 대기에서 중단했다면 커널을 재시작한 후 처음부터 실행하세요.

In [ ]:
RUN_LIVE = True
SCENARIO = 0  # 0: 예산/맛, 1: 공부 목적, 2: 저렴한 라떼, 3: 결과 없음

scenarios = [
    ('근처에서 5천 원 이하의 달지 않은 커피 추천해 줘.',
     SearchFilters(origin=origin, radius_m=500, max_price_krw=5000,
                   sweetness='NOT_SWEET', sort_by='price')),
    ('공부하기 좋은 카페에서 마실 음료 추천해 줘.',
     SearchFilters(origin=origin, radius_m=500, required_cafe_tags=('STUDY',), sort_by='price')),
    ('500m 안에서 5천 원 이하의 저렴한 라떼 찾아줘.',
     SearchFilters(origin=origin, radius_m=500, max_price_krw=5000,
                   menu_name='라떼', sort_by='price')),
    ('500m 안에서 천 원 이하 커피 찾아줘. 예산을 바꾸지 마.',
     SearchFilters(origin=origin, radius_m=500, max_price_krw=1000, sort_by='price')),
]
message, confirmed = scenarios[SCENARIO]
print('요청:', message)
print('확정 조건:', asdict(confirmed))

model = None
gemini_settings = None
if RUN_LIVE:
    from app.llm.settings import read_gemini_settings
    gemini_settings = read_gemini_settings(AI_ROOT)
    print('설정 확인 완료. API 호출은 아직 하지 않았습니다. 4번 셀을 실행하세요.', flush=True)
else:
    print('실제 호출 비활성화: RUN_LIVE=True로 변경하면 실행 가능합니다.')


## 4. 실제 도구 호출 실행

LLM이 검색 방법과 추가 조회를 결정합니다. 결과의 가격·메뉴는 Python이 조회 자료에서 가져옵니다. `QUESTION`이면 질문을 검토하고, 동의한 조건만 직접 수정한 뒤 재실행하세요. 단순히 '예'라고 답했다고 자동 완화하지 않습니다.

In [ ]:
state = ConversationState()  # 이 셀을 다시 실행하면 대화가 초기화됩니다.
live_result = None
if RUN_LIVE:
    if gemini_settings is None:
        raise RuntimeError('3번 설정 셀을 먼저 실행하세요.')
    from app.llm.gemini import GeminiModel
    print('Gemini 클라이언트 초기화 중...', flush=True)
    if model is not None:
        model.close()
    model = GeminiModel(api_key=gemini_settings[0], model=gemini_settings[1])
    print('Gemini 요청 시작 (턴 시간 예산 60초)...', flush=True)
    live_result = run_turn(model, repo, message, constraints=confirmed, state=state)
    show_result(live_result)
    if live_result.status == 'RESULT':
        assert all(m.offer.evidence.is_mock for m in live_result.recommendations)
        if confirmed.max_price_krw is not None:
            assert all(m.offer.price_krw <= confirmed.max_price_krw for m in live_result.recommendations)
else:
    print('건너뜀: 실제 API 호출 없음.')

## 5. 후속 질문: 그중에서 가까운 곳은?

`within_previous=True`로 직전 추천 판매 옵션 범위를 유지하고, 확정 조건은 그대로 전달합니다. 세션은 메모리에만 있습니다. 결과가 없거나 질문 응답이면 먼저 이전 요청을 해결하세요.

In [ ]:
if RUN_LIVE and live_result is not None and live_result.status == 'RESULT' and state.last_offer_ids:
    followup = run_turn(model, repo, '그중에서 가까운 곳은?',
                        constraints=replace(confirmed, sort_by='distance'),
                        state=state, within_previous=True)
    show_result(followup)
else:
    print('후속 호출 건너뜀.')

## 6. 종료와 기록

저장·공유 전에 노트북 출력을 지우세요. 커밋된 노트북은 출력 없이 제공합니다. `ERROR`는 API/조회 실패, `LIMIT`는 실행 예산 소진이며 검색 결과 없음과 다릅니다.

이 노트북의 실제 호출 결과는 실행 전까지 미검증입니다. 평가 시 조건 만족 여부, 실제 도구 호출 여부, 허위 ID 거부, 호출 수와 시간을 확인하세요. 모델의 일반 문장 품질·자동 조건 추출은 다음 단계의 평가 대상입니다.

### 오류 진단
코드 변경 후에는 커널을 재시작하고 처음부터 실행하세요. 오류 trace에 `http_status`와 `error_code`가 표시됩니다. HTTP 429는 할당량, 404는 모델/리소스, 403은 권한, 504는 응답 시간 초과입니다. `LOCAL_SDK_ERROR`는 로컬 SDK 또는 응답 처리 문제입니다. 원본 오류와 API 키는 출력하지 않습니다.

무료 티어에서도 모델별·프로젝트별 한도가 적용됩니다. 실패 시 연속 전체 실행 대신 오류 코드부터 확인하세요. 이번 진단 호출은 20초 제한에서 HTTP 504였으며, 이전 노트북 오류의 원인이 동일한지는 확인되지 않았습니다.


In [ ]:
if model is not None:
    model.close()
    model = None
print('완료. 다시 실제 호출하려면 설정 셀부터 실행하세요.')